In [1]:
import numpy as np
import time
import sys
import os

import numpy as np
import yfinance as fy
import QuantLib as ql
from typing import Optional, Union
import pandas as pd
import matplotlib.pyplot as plt
import multiprocess
import seaborn as sb
from tqdm import tqdm

from inception.instruments.constant_parameters import NOTES_PARAMETERS
from inception.utils import timer
from inception.quantlib_python import date_to_date
from inception.instruments import ReferenceMultiIndexReturnSimulator, MultiAssetAutoCallableNote

import mocaxextendpy.mocax_extend as me

In [2]:
parameters = NOTES_PARAMETERS['CIBC Multi Assets One']
start_date =  pd.Timestamp('2024-02-14')
days_interval = [(pd.Timestamp(d) - start_date).days for d in parameters['valuation dates']]
days_periods = list(zip([0] + [d + 1 for d in days_interval[:-1]], days_interval))
days_periods

[(0, 90),
 (91, 182),
 (183, 274),
 (275, 366),
 (367, 455),
 (456, 547),
 (548, 639),
 (640, 734),
 (735, 820),
 (821, 912),
 (913, 1006),
 (1007, 1098),
 (1099, 1185),
 (1186, 1279),
 (1280, 1370),
 (1371, 1461)]

In [3]:
def autocallable_note_pricer(x: list, additonal_data = None):
    """
    The Autocallable Notes pricer function wrapper, (adapts the original function signature to the signature expected by MoCaX)
    :param x: [stock A price, stock B price, stock C price, days after the start date]
    :param additional_data: extra parameters required by MoCaX
    """
    import numpy as np
    import pandas as pd
    from inception.instruments.constant_parameters import NOTES_PARAMETERS
    from inception.instruments import ReferenceMultiIndexReturnSimulator, MultiAssetAutoCallableNote
    
    start_date = '2024-02-14'
    # Input parameters:
    spot_price = np.array(x[:3])
    eval_date = pd.Timestamp(start_date) + pd.Timedelta(days=x[3])
    
    corr_matrix = np.array(
        [[1.        , 0.96432854, 0.80799003],
        [0.96432854, 1.        , 0.8267037 ],
        [0.80799003, 0.8267037 , 1.        ]]
    )
    issued_price = [382.82, 494.08, 142.86]
    index_vol = [0.16, 0.16, 0.16]
    risk_free_rate = 0.05
    path_num = 100_000
    d_s = 0.02
    parameters = NOTES_PARAMETERS['CIBC Multi Assets One']
    simulator = ReferenceMultiIndexReturnSimulator(issued_price, spot_price, risk_free_rate, index_vol, corr_matrix, d_s)
    notes = MultiAssetAutoCallableNote(eval_date,
                                       simulator, 
                                       parameters['valuation dates'],
                                       parameters['coupon dates'],
                                       parameters['call dates'],
                                       coupon_barrier=parameters['coupon barrier'],
                                       call_barrier=parameters['call barrier'],
                                       principal_barrier=parameters['principal barrier'],
                                       coupon_amount=parameters['coupon amount'],
                                       free_rate=risk_free_rate,
                                       n_paths=path_num)

    
    return notes.value

In [4]:
autocallable_note_pricer([382.82, 494.08, 142.86, 0])

101.7055299866333

In [5]:
def autocallable_notes_chebyshev_tensor_extension_approximate(time_bucket: int, pricer, num_scenarios: int = 2000):
    """
    Using Chebyshev Tensor extension algorithm to approximate the Autocallable notes pricer
    :param time_bucket: the time period index
    :param num_scenarios: Number of subgrid
    :return:
    """
    from inception.instruments.constant_parameters import NOTES_PARAMETERS
    import mocaxextendpy.mocax_extend as me
    import pandas as pd
    import numpy as np
    import os
    from tqdm import tqdm
    
    parameters = NOTES_PARAMETERS['CIBC Multi Assets One']
    start_date =  pd.Timestamp('2024-02-14')
    days_interval = [(pd.Timestamp(d) - start_date).days for d in parameters['valuation dates']]
    days_periods = list(zip([0] + [d + 1 for d in days_interval[:-1]], days_interval))
    
    # Number of dimensions
    num_dimensions = 4

    # MoCaX accuracy parameters, Chebyshev Nodes
    n_nodes = [15, 15, 15, 15]

    # Function domain for stock A, B and C price, Time to Maturity
    issued_price = [382.82, 494.08, 142.86]

    lower_bound = 0.2
    upper_bound = 1.5

    domain_values = [
        [issued_price[0] * lower_bound, issued_price[0] * upper_bound] ,  # Stock A price
        [issued_price[1] * lower_bound, issued_price[1] * upper_bound] ,  # Stock B price
        [issued_price[2] * lower_bound, issued_price[2] * upper_bound] ,  # Stock C price
        list(days_periods[time_bucket])                                   # Days after the start date
    ]


    mocax_extend = me.MocaxExtend(num_dimensions, n_nodes, domain_values)

    # -- Subgrid creation
    random_cheb_points = mocax_extend.subgrid_by_number(num_scenarios)

    values_subgrid = []
    for x in tqdm(random_cheb_points):
        values_subgrid.append(pricer(x))

    # -- Incorporate data to rank adaptive object
    mocax_extend.set_subgrid_values(values_subgrid)
    mocax_extend.gen_train_val_data()

    # -- Size comparison of original grid with subgrid
    original_grid_size = mocax_extend.get_tensor_size()
    subgrid_size = mocax_extend.get_subgrid_size()
    print("Subgrid only {:.5f}% of original grid\n".format(100 * subgrid_size / original_grid_size))

    """ Rank adaptive algorithm run """

    # -- Parameters defined
    rank_adaptive_params = {"tolerance": 1e-3,
                            "rel_tolerance": 1e-8,
                            "max_iters": 100,
                            "max_rank": 10,
                            "print_progress": True,
                            "max_rounds": 5}


    # -- Training of Chebyshev Tensor in TT format
    mocax_extend.run_rank_adaptive_algo(**rank_adaptive_params)


    """ Serialization and deserialization """

    ## -- Serialization
    file_dir = f'./chebyshev_database/'
    if not os.path.exists(file_dir):
        os.makedirs(file_dir)
        
    file_name = file_dir + f'autocallable_tensor_extension_{time_bucket}.pickle'
    mocax_extend.serialize(file_name)

In [1]:
import numpy as np

In [9]:
# --------------------------------------------------
# time_bucket_1 = (0, autocallable_note_pricer, 5000)
# time_bucket_2 = (1, autocallable_note_pricer, 5000)
# time_bucket_3 = (2, autocallable_note_pricer, 5000)
# time_bucket_4 = (3, autocallable_note_pricer, 5000)

# time_bucket_1 = (4, autocallable_note_pricer, 5000)
# time_bucket_2 = (5, autocallable_note_pricer, 5000)
# time_bucket_3 = (6, autocallable_note_pricer, 5000)
# time_bucket_4 = (7, autocallable_note_pricer, 5000)

# time_bucket_1 = (8, autocallable_note_pricer, 5000)
# time_bucket_2 = (9, autocallable_note_pricer, 5000)
# time_bucket_3 = (10, autocallable_note_pricer, 5000)
# time_bucket_4 = (11, autocallable_note_pricer, 5000)

time_bucket_1 = (12, autocallable_note_pricer, 5000)
time_bucket_2 = (13, autocallable_note_pricer, 5000)
time_bucket_3 = (14, autocallable_note_pricer, 5000)
# --------------------------------------------------

p1 = multiprocess.Process(target=autocallable_notes_chebyshev_tensor_extension_approximate, args=time_bucket_1)
p2 = multiprocess.Process(target=autocallable_notes_chebyshev_tensor_extension_approximate, args=time_bucket_2)
p3 = multiprocess.Process(target=autocallable_notes_chebyshev_tensor_extension_approximate, args=time_bucket_3)
# p4 = multiprocess.Process(target=autocallable_notes_chebyshev_tensor_extension_approximate, args=time_bucket_4)

# starting process 1
p1.start()
# starting process 2
p2.start()
# starting process 3
p3.start()
# starting process 3
# p4.start()

# wait until process 1 is finished
p1.join()
# wait until process 2 is finished
p2.join()
p3.join()
# p4.join()

# both processes finished
print("Done!")

Done!
